# Play MASA Safety Gridworlds

Choose an environment with the buttons. The preview cell renders inline with `rgb_array`, and the last cell opens a `pygame` window for keyboard play.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

VALID_ENV_NAMES = ("island_navigation", "conveyor_belt", "sokoban")

def make_env(env_name, **kwargs):
    if env_name == "island_navigation":
        from masa.envs.discrete.island_navigation import IslandNavigation

        return IslandNavigation(**kwargs)
    if env_name == "conveyor_belt":
        from masa.envs.discrete.conveyor_belt import ConveyorBelt

        return ConveyorBelt(**kwargs)
    if env_name == "sokoban":
        from masa.envs.discrete.sokoban import Sokoban

        return Sokoban(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="sokoban",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset info:", info)
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def play_env(env_name=None, seed: int = 0):
    import pygame

    if env_name is None:
        env_name = ENV_SELECTOR.value
    action_keys = {
        pygame.K_RIGHT: 0,
        pygame.K_UP: 1,
        pygame.K_LEFT: 2,
        pygame.K_DOWN: 3,
        pygame.K_d: 0,
        pygame.K_w: 1,
        pygame.K_a: 2,
        pygame.K_s: 3,
    }
    env = make_env(env_name, render_mode="human", render_window_size=512)
    obs, info = env.reset(seed=seed)
    finished = False
    print("Controls: arrows or WASD move, R resets, Q or Escape quits.")
    print("reset info:", info)

    try:
        while True:
            action = None
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    return
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    return
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    print("reset info:", info)
                    continue
                if event.key in action_keys and not finished:
                    action = action_keys[event.key]

            if action is not None:
                obs, reward, terminated, truncated, info = env.step(action)
                finished = terminated or truncated
                print({
                    "reward": reward,
                    "terminated": terminated,
                    "truncated": truncated,
                    "info": info,
                })

            pygame.time.wait(16)
    finally:
        env.close()

play_env(seed=SEED)
